In [ ]:
import tensorflow as tf
import numpy as np
from keras import layers, optimizers, callbacks
from transformers import GPT2TokenizerFast
from keras.saving import register_keras_serializable


# Load text
with open('/content/Data_2.txt','r',encoding='utf-8') as f:
    text = f.read()

seq_len = 100
# ============================================
# GPT-2 Tokenization Setup
# ============================================
print("Initializing GPT-2 tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

vocab_size = len(tokenizer)
print(f"Vocabulary size: {vocab_size}")

def encode(text):
    """Encode text using GPT-2 tokenizer"""
    return tokenizer.encode(text, add_special_tokens=False)

def decode(indices):
    """Decode token indices back to text"""
    return tokenizer.decode(indices)
# Encode the entire text
print("Encoding text...")
encoded = encode(text)

In [ ]:

# Encode full text
tokens = tf.constant(encoded, dtype=tf.int32)

# Trim to a clean multiple of (seq_len + 1)
tokens = tokens[: (len(tokens) // (seq_len + 1)) * (seq_len + 1)]

# Reshape into windows
windows = tf.reshape(tokens, (-1, seq_len + 1))

# Dataset before mapping
ds = tf.data.Dataset.from_tensor_slices(windows)

# Input/target split
ds = ds.map(lambda x: (x[:-1], x[1:]),
            num_parallel_calls=tf.data.AUTOTUNE)

# ============================================
# Train / Validation Split
# ============================================

total_sequences = windows.shape[0]
val_size = int(total_sequences * 0.1)   # 10% val
train_size = total_sequences - val_size

train_ds = ds.take(train_size)
val_ds   = ds.skip(train_size)

# ============================================
# Batch + Prefetch
# ============================================

BATCH = 32

train_ds = (
    train_ds
    .shuffle(10000)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:

initializer = tf.keras.initializers.TruncatedNormal(stddev=0.02)
ln_eps = 1e-5  # GPT-2 uses small eps

d_model = 512
heads = 6
n_layers = 4
dropout = 0.2


initializer = tf.keras.initializers.TruncatedNormal(stddev=0.02)
ln_eps = 1e-5  # GPT-2 uses small eps

def transformer_block(x, d_model, heads, dropout):
    # ---- Attention block (Pre-LN) ----
    ln1 = layers.RMSNormalization(epsilon=ln_eps)(x)
    att = layers.MultiHeadAttention(
        num_heads=heads,
        key_dim=d_model // heads,
        dropout=dropout,
        kernel_initializer=initializer,
        output_shape=d_model
    )(ln1, ln1, ln1, use_causal_mask=True)
    att = layers.Dropout(dropout)(att)
    x = x + att

    # ---- Feed Forward block (Pre-LN) ----
    ln2 = layers.RMSNormalization(epsilon=ln_eps)(x)
    ffn = layers.Dense(d_model * 4, activation='gelu', use_bias=True,
                       kernel_initializer=initializer)(ln2)
    #ffn = layers.Dropout(dropout)(ffn)
    ffn = layers.Dense(d_model, use_bias=True, kernel_initializer=initializer)(ffn)
    #ffn = layers.Dropout(dropout)(ffn)
    x = x + ffn

    return x

@register_keras_serializable()
class PositionEmbedding(layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=d_model,
                                        embeddings_initializer=initializer)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(0, seq_len, dtype=tf.int32)[tf.newaxis, :]  # (1, seq)
        return self.pos_emb(positions)  # broadcasts over batch


# ---------- Build model (token embedding layer saved for weight tying) ----------
inputs = layers.Input(shape=(seq_len,), dtype=tf.int32)

# Keep a reference to the embedding layer so we can tie weights later
token_embedding_layer = layers.Embedding(
    input_dim=vocab_size,
    output_dim=d_model,
    embeddings_initializer=initializer,
    mask_zero=False,  # don't use mask with causal LM
    name="token_embedding"
)

token_emb = token_embedding_layer(inputs)

pos_emb_layer = PositionEmbedding(max_len=seq_len, d_model=d_model)
pos_emb = pos_emb_layer(inputs)

x = token_emb + pos_emb
x = layers.Dropout(dropout)(x)

for i in range(n_layers):
    x = transformer_block(x, d_model, heads, dropout)

x = layers.RMSNormalization(epsilon=ln_eps)(x)
emb_weights = token_embedding_layer.embeddings  # variable

def lm_head(x):
    return tf.matmul(x, emb_weights, transpose_b=True)

output = layers.Lambda(lm_head, name="logits")(x)

model = tf.keras.Model(inputs, output)

# ---------- Optimizer: use GPT-like defaults ----------
opt = optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.1,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-8,
    clipnorm=1.0
)

model.compile(
    optimizer=opt,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

class PeriodicCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, save_every_n_steps=400, filepath_prefix='checkpoint_step'):
        super().__init__()
        self.save_every_n_steps = save_every_n_steps
        self.filepath_prefix = filepath_prefix
        self.step_count = 0

    def on_batch_end(self, batch, logs=None):
        self.step_count += 1

        if self.step_count % self.save_every_n_steps == 0:
            filepath = f"{self.filepath_prefix}_{self.step_count}.weights.h5"
            self.model.save_weights(filepath, overwrite=True)
            print(f"\nSaved weights to {filepath}")
model.load_weights('/content/drive/MyDrive/checkpoint_step_10000.weights (1).weights.h5')

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    verbose=1,
    callbacks=[PeriodicCheckpoint(save_every_n_steps=5000)]
)


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=1.0, top_k=None):
    # Determine padding token
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else \
             (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0)

    # Encode prompt
    input_ids = tokenizer.encode(prompt)

    for _ in range(max_new_tokens):
        # Pad to seq_len if shorter
        if len(input_ids) < seq_len:
            inp = pad_sequences([input_ids], maxlen=seq_len, padding='pre', truncating='pre', value=pad_id)
        else:
            inp = [input_ids[-seq_len:]]  # keep last seq_len tokens

        inp = tf.constant(inp, dtype=tf.int32)

        # Forward pass
        logits = model(inp, training=False)[:, -1, :]  # (1, vocab)

        # Temperature scaling
        logits = logits / max(temperature, 1e-9)

        # Top-k filtering
        if top_k is not None and top_k > 0:
            k = min(top_k, logits.shape[-1])
            values, _ = tf.math.top_k(logits, k=k)
            cutoff = values[:, -1]
            logits = tf.where(logits < cutoff[:, None], tf.constant(-1e10, dtype=logits.dtype), logits)

        # Sample next token
        next_id = tf.random.categorical(logits, num_samples=1)
        next_id = tf.cast(next_id, tf.int32)
        input_ids.append(int(next_id[0,0]))

    # Decode
    return tokenizer.decode(input_ids)

output = generate(
    model,
    tokenizer,
    prompt="“I’ve been watching a new story,” she said, “It's a very good idea.”",
    max_new_tokens=50,
    temperature=0.5,
    top_k=40
)

print(output)
